In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ==========================================
# 1. データ読み込み
# ==========================================

DATA_PATH = "../../data/raw/PPI_resilient_all.csv"

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    low_memory=False
)

# 保存先
OUTPUT = Path("results/descriptive_statistics")
OUTPUT.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. 基本情報
# ==========================================

print("="*70)
print("Data Shape")
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nData Types")
print(df.dtypes)

print("\nMissing Values")
print(df.isnull().sum())

# ==========================================
# 3. 数値・カテゴリ列
# ==========================================

numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

print(f"\nNumeric columns : {len(numeric_cols)}")
print(f"Categorical columns : {len(categorical_cols)}")

# ==========================================
# 4. 数値変数の記述統計
# ==========================================

numeric_summary = df[numeric_cols].describe().T

numeric_summary["median"] = df[numeric_cols].median()
numeric_summary["missing"] = df[numeric_cols].isna().sum()
numeric_summary["missing_rate"] = (
    df[numeric_cols].isna().mean()*100
)

numeric_summary.to_csv(
    OUTPUT/"numeric_summary.csv",
    encoding="utf-8-sig"
)

print("\nNumeric Summary")
print(numeric_summary)

# ==========================================
# 5. カテゴリ変数の記述統計
# ==========================================

rows = []

for col in categorical_cols:

    mode = df[col].mode(dropna=True)

    rows.append({
        "Variable": col,
        "Count": df[col].count(),
        "Missing": df[col].isna().sum(),
        "MissingRate(%)": df[col].isna().mean()*100,
        "Unique": df[col].nunique(),
        "Top": mode.iloc[0] if len(mode)>0 else np.nan,
        "TopFreq": df[col].value_counts(dropna=False).iloc[0]
    })

categorical_summary = pd.DataFrame(rows)

categorical_summary.to_csv(
    OUTPUT/"categorical_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================
# 6. 全列サマリー
# ==========================================

summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_rate": df.isna().mean()*100,
    "unique": df.nunique()
})

summary.to_csv(
    OUTPUT/"all_columns_summary.csv",
    encoding="utf-8-sig"
)

# ==========================================
# 7. 欠損率の可視化
# ==========================================

missing = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      *100
)

plt.figure(figsize=(12,10))

missing.plot.bar()

plt.ylabel("Missing Rate (%)")
plt.title("Missing Rate by Variable")
plt.tight_layout()

plt.savefig(
    OUTPUT/"missing_rate.png",
    dpi=300
)

plt.close()

# ==========================================
# 8. 数値変数ヒストグラム
# ==========================================

hist_dir = OUTPUT/"histograms"
hist_dir.mkdir(exist_ok=True)

for col in numeric_cols:

    plt.figure(figsize=(6,4))

    df[col].dropna().hist(
        bins=30
    )

    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.tight_layout()

    plt.savefig(
        hist_dir/f"{col}.png",
        dpi=300
    )

    plt.close()

# ==========================================
# 9. カテゴリ変数棒グラフ（上位10）
# ==========================================

bar_dir = OUTPUT/"categorical"
bar_dir.mkdir(exist_ok=True)

for col in categorical_cols:

    vc = df[col].value_counts().head(10)

    plt.figure(figsize=(8,5))

    vc.sort_values().plot.barh()

    plt.title(col)
    plt.xlabel("Count")

    plt.tight_layout()

    plt.savefig(
        bar_dir/f"{col}.png",
        dpi=300
    )

    plt.close()

# ==========================================
# 10. 年別プロジェクト数
# ==========================================

if "Financial closure year" in df.columns:

    plt.figure(figsize=(10,5))

    (
        df["Financial closure year"]
        .value_counts()
        .sort_index()
        .plot()
    )

    plt.title("Projects by Financial Closure Year")
    plt.xlabel("Year")
    plt.ylabel("Projects")

    plt.tight_layout()

    plt.savefig(
        OUTPUT/"projects_by_year.png",
        dpi=300
    )

    plt.close()

# ==========================================
# 11. セクター別件数
# ==========================================

if "Primary sector" in df.columns:

    plt.figure(figsize=(8,5))

    (
        df["Primary sector"]
        .value_counts()
        .plot.bar()
    )

    plt.ylabel("Projects")
    plt.title("Projects by Primary Sector")

    plt.xticks(rotation=45)

    plt.tight_layout()

    plt.savefig(
        OUTPUT/"primary_sector.png",
        dpi=300
    )

    plt.close()

# ==========================================
# 12. 地域別件数
# ==========================================

if "Region" in df.columns:

    plt.figure(figsize=(8,5))

    (
        df["Region"]
        .value_counts()
        .plot.bar()
    )

    plt.ylabel("Projects")
    plt.title("Projects by Region")

    plt.xticks(rotation=45)

    plt.tight_layout()

    plt.savefig(
        OUTPUT/"region.png",
        dpi=300
    )

    plt.close()

# ==========================================
# 13. 所得水準別件数
# ==========================================

if "IncomeGroup" in df.columns:

    plt.figure(figsize=(8,5))

    (
        df["IncomeGroup"]
        .value_counts()
        .plot.bar()
    )

    plt.ylabel("Projects")
    plt.title("Projects by Income Group")

    plt.xticks(rotation=30)

    plt.tight_layout()

    plt.savefig(
        OUTPUT/"income_group.png",
        dpi=300
    )

    plt.close()

# ==========================================
# 14. PPIタイプ別件数
# ==========================================

if "Type of PPI" in df.columns:

    plt.figure(figsize=(8,5))

    (
        df["Type of PPI"]
        .value_counts()
        .plot.bar()
    )

    plt.ylabel("Projects")
    plt.title("Projects by PPI Type")

    plt.xticks(rotation=30)

    plt.tight_layout()

    plt.savefig(
        OUTPUT/"ppi_type.png",
        dpi=300
    )

    plt.close()

print("\n" + "="*70)
print("Finished!")
print(f"Results saved in: {OUTPUT.resolve()}")
print("="*70)

Data Shape
(8161, 45)

Columns
['Region', 'Country', 'IncomeGroup', 'IDA Status', 'Financial closure year', 'Financial closure Month', 'Project name', 'RelatedNames', 'Type of PPI', 'Subtype of PPI', 'Project status', 'Primary sector', 'Subsector', 'Segment', 'Location', 'ContractPeriod', 'GovtGrantingContract', 'DirectGovtSupport', 'DirectGovtSupportValue', 'InDirectGovtSupport', 'InDirectGovtSupportValue', 'Total Equity', 'InvestmentYear', 'PercentPrivate', 'FeesToGovernment', 'PhysicalAssets', 'TotalInvestment', 'CapacityType', 'Capacity', 'Technology', 'RelatedProjects', 'BidCriteria', 'AwardMethod', 'NumberOfBids', 'Sponsors', 'Sponsors Country', 'Main Revenue Source', 'Other Revenue Source', 'MultiLateralSupport', 'BiLateralSupport', 'TotalDebtFunding', 'DebtEquityGrantRatio', 'ProjectBanks', 'UnsolicitedProposal', 'PublicDisclosure']

Data Types
Region                        str
Country                       str
IncomeGroup                   str
IDA Status                    str

In [3]:
# ============================================================
# GDP per Capita × PPI Visualization
# Part 1 : Data Loading + Common Functions
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# matplotlib settings
# ------------------------------------------------------------

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300

sns.set_theme(style="whitegrid")

# ------------------------------------------------------------
# Path
# ------------------------------------------------------------

GDP_PATH = "../../data/raw/GDP per capita/db9758b4-95e5-4584-b366-5dd38a5d3769_Data.csv"

PPI_PATH = "../../data/raw/PPI_resilient_all.csv"

OUTPUT_DIR = Path("../../results/gdp_visualization")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Read GDP data
# ------------------------------------------------------------

print("=" * 70)
print("Loading GDP per Capita data...")

gdp = pd.read_csv(GDP_PATH)

# GDP year columns
year_cols = [c for c in gdp.columns if "[YR" in c]

# convert to numeric
gdp[year_cols] = (
    gdp[year_cols]
    .replace("..", np.nan)
    .apply(pd.to_numeric, errors="coerce")
)

# ------------------------------------------------------------
# 2. Average GDP per capita
# ------------------------------------------------------------

country_gdp = (
    gdp[
        [
            "Country Name",
            "Country Code"
        ] + year_cols
    ]
    .copy()
)

country_gdp["avg_gdp_pc"] = (
    country_gdp[year_cols]
    .mean(axis=1)
)

country_gdp = country_gdp[
    [
        "Country Name",
        "Country Code",
        "avg_gdp_pc"
    ]
]

country_gdp.columns = [
    "country_name",
    "country_iso",
    "avg_gdp_pc"
]

print(country_gdp.head())

# ------------------------------------------------------------
# 3. GDP Quintile
# ------------------------------------------------------------

country_gdp = country_gdp.dropna(
    subset=["avg_gdp_pc"]
)

country_gdp["gdp_group"] = pd.qcut(
    country_gdp["avg_gdp_pc"],
    q=5,
    labels=[
        "Very Low",
        "Low",
        "Middle",
        "High",
        "Very High"
    ]
)

print("\nGDP Quintile")
print(country_gdp["gdp_group"].value_counts())

# ------------------------------------------------------------
# 4. Read PPI data
# ------------------------------------------------------------

print("=" * 70)
print("Loading PPI data...")

ppi = pd.read_csv(
    PPI_PATH,
    low_memory=False
)

print(ppi.shape)

# ------------------------------------------------------------
# 5. Merge GDP
# ------------------------------------------------------------
#
# 基本はCountry Name同士で結合
#
# 必要ならCountry Codeに変更してください
#

ppi = ppi.merge(
    country_gdp,
    left_on="Country",
    right_on="country_name",
    how="left"
)

print("\nMerged Shape")
print(ppi.shape)

print("\nGDP Missing")

print(
    ppi["gdp_group"]
    .isna()
    .mean()
)

# ------------------------------------------------------------
# 6. Common aggregation function
# ------------------------------------------------------------

def create_share_table(
    df,
    feature,
    group="gdp_group",
    top_n=15
):
    """
    feature × GDP group

    Returns
    -------
    counts
    shares
    lift
    """

    tmp = df[
        [feature, group]
    ].dropna()

    counts = (
        tmp
        .groupby(
            [feature, group]
        )
        .size()
        .unstack(fill_value=0)
    )

    # Top N

    if len(counts) > top_n:

        top = (
            counts.sum(axis=1)
            .sort_values(ascending=False)
            .head(top_n)
            .index
        )

        counts = counts.loc[top]

    # Column percentage

    shares = counts.div(
        counts.sum(axis=0),
        axis=1
    ) * 100

    # Lift

    lift = shares.div(
        shares.mean(axis=1),
        axis=0
    )

    return counts, shares, lift

# ------------------------------------------------------------
# 7. Heatmap
# ------------------------------------------------------------

def plot_heatmap(
    shares,
    feature,
    save_dir
):

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        shares,
        annot=True,
        fmt=".1f",
        cmap="YlGnBu",
        linewidths=.5
    )

    plt.title(
        f"{feature} by GDP per Capita Group (%)",
        fontsize=15
    )

    plt.xlabel("GDP Quintile")

    plt.ylabel(feature)

    plt.tight_layout()

    plt.savefig(
        save_dir / f"{feature}_heatmap.png"
    )

    plt.close()

# ------------------------------------------------------------
# 8. Stacked bar chart
# ------------------------------------------------------------

def plot_stacked_bar(
    shares,
    feature,
    save_dir
):

    fig, ax = plt.subplots(
        figsize=(11, 6)
    )

    shares.T.plot(
        kind="bar",
        stacked=True,
        ax=ax
    )

    ax.set_ylabel("% of Projects")

    ax.set_xlabel("GDP Quintile")

    ax.set_title(
        f"{feature} Composition"
    )

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.tight_layout()

    plt.savefig(
        save_dir / f"{feature}_stacked_bar.png"
    )

    plt.close()

# ------------------------------------------------------------
# 9. Lift Heatmap
# ------------------------------------------------------------

def plot_lift(
    lift,
    feature,
    save_dir
):

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        lift,
        cmap="RdBu_r",
        center=1,
        linewidths=.5
    )

    plt.title(
        f"Overrepresentation of {feature}",
        fontsize=15
    )

    plt.xlabel("GDP Quintile")

    plt.ylabel(feature)

    plt.tight_layout()

    plt.savefig(
        save_dir / f"{feature}_lift.png"
    )

    plt.close()

# ------------------------------------------------------------
# 10. Complete visualization function
# ------------------------------------------------------------

def visualize_feature(
    df,
    feature,
    output_dir=OUTPUT_DIR,
    top_n=15
):

    print(f"\nProcessing : {feature}")

    counts, shares, lift = create_share_table(
        df,
        feature,
        top_n=top_n
    )

    feature_dir = output_dir / feature.replace("/", "_")
    feature_dir.mkdir(
        exist_ok=True
    )

    counts.to_csv(
        feature_dir / "counts.csv",
        encoding="utf-8-sig"
    )

    shares.to_csv(
        feature_dir / "shares.csv",
        encoding="utf-8-sig"
    )

    lift.to_csv(
        feature_dir / "lift.csv",
        encoding="utf-8-sig"
    )

    plot_heatmap(
        shares,
        feature,
        feature_dir
    )

    plot_stacked_bar(
        shares,
        feature,
        feature_dir
    )

    plot_lift(
        lift,
        feature,
        feature_dir
    )

    print("Done.")

Loading GDP per Capita data...
     country_name country_iso    avg_gdp_pc
0     Afghanistan         AFG    456.053629
1         Albania         ALB   2927.046596
2         Algeria         DZA   3631.825374
3  American Samoa         ASM  12732.337240
4         Andorra         AND  35722.299250

GDP Quintile
gdp_group
Very Low     53
Low          52
Middle       52
High         52
Very High    52
Name: count, dtype: int64
Loading PPI data...
(8161, 45)

Merged Shape
(8161, 49)

GDP Missing
0.05967405955152555


In [5]:
# ============================================================
# Part 2 : Automatic Visualization
# ============================================================

print("=" * 70)
print("Generating GDP × PPI visualizations...")
print("=" * 70)

FEATURES = [
    "Primary sector",
    "Subtype of PPI",
    "Technology",
    "Project status",
    "Main Revenue Source"
]

for feature in FEATURES:

    print("\n" + "=" * 60)
    print(f"Processing : {feature}")
    print("=" * 60)

    # ------------------------------
    # column check
    # ------------------------------

    if feature not in ppi.columns:
        print(f"Column '{feature}' does not exist.")
        continue

    tmp = ppi[[feature, "gdp_group"]].dropna()

    if tmp.empty:
        print("No observations.")
        continue

    # ------------------------------
    # safe folder name
    # ------------------------------

    safe_name = (
        feature
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "")
        .replace("*", "")
        .replace("?", "")
        .replace('"', "")
        .replace("<", "")
        .replace(">", "")
        .replace("|", "")
    )

    feature_dir = OUTPUT_DIR / safe_name
    feature_dir.mkdir(parents=True, exist_ok=True)

    try:

        # --------------------------
        # aggregation
        # --------------------------

        counts, shares, lift = create_share_table(
            tmp,
            feature,
            top_n=15
        )

        # --------------------------
        # save csv
        # --------------------------

        counts.to_csv(
            feature_dir / "counts.csv",
            encoding="utf-8-sig"
        )

        shares.to_csv(
            feature_dir / "shares.csv",
            encoding="utf-8-sig"
        )

        lift.to_csv(
            feature_dir / "lift.csv",
            encoding="utf-8-sig"
        )

        # --------------------------
        # plots
        # --------------------------

        plot_heatmap(
            shares,
            safe_name,
            feature_dir
        )

        plot_stacked_bar(
            shares,
            safe_name,
            feature_dir
        )

        plot_lift(
            lift,
            safe_name,
            feature_dir
        )

        print(f"Saved -> {feature_dir.resolve()}")

    except Exception as e:

        print(f"Error while processing '{feature}'")
        print(e)

# ============================================================
# Summary
# ============================================================

print("\n")
print("=" * 70)
print("Completed")
print("=" * 70)

print("Output directory")
print(OUTPUT_DIR.resolve())

print("\nGenerated files")

for feature in FEATURES:

    safe_name = (
        feature
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "")
        .replace("*", "")
        .replace("?", "")
        .replace('"', "")
        .replace("<", "")
        .replace(">", "")
        .replace("|", "")
    )

    folder = OUTPUT_DIR / safe_name

    if folder.exists():

        print(f"\n{folder.name}")

        for f in sorted(folder.iterdir()):
            print("   ", f.name)

print("\nDone.")

Generating GDP × PPI visualizations...

Processing : Primary sector
Saved -> C:\Users\81809\OneDrive\ドキュメント\GitHub\DSPP-Infra\results\gdp_visualization\Primary_sector

Processing : Subtype of PPI
Saved -> C:\Users\81809\OneDrive\ドキュメント\GitHub\DSPP-Infra\results\gdp_visualization\Subtype_of_PPI

Processing : Technology
Saved -> C:\Users\81809\OneDrive\ドキュメント\GitHub\DSPP-Infra\results\gdp_visualization\Technology

Processing : Project status
Saved -> C:\Users\81809\OneDrive\ドキュメント\GitHub\DSPP-Infra\results\gdp_visualization\Project_status

Processing : Main Revenue Source
Saved -> C:\Users\81809\OneDrive\ドキュメント\GitHub\DSPP-Infra\results\gdp_visualization\Main_Revenue_Source


Completed
Output directory
C:\Users\81809\OneDrive\ドキュメント\GitHub\DSPP-Infra\results\gdp_visualization

Generated files

Primary_sector
    counts.csv
    lift.csv
    Primary sector_heatmap.png
    Primary sector_lift.png
    Primary sector_stacked_bar.png
    Primary_sector_heatmap.png
    Primary_sector_lift.png
 

In [6]:
# Technologyのユニーク値一覧
technology_list = (
    ppi["Technology"]
    .dropna()
    .sort_values()
    .unique()
)

for tech in technology_list:
    print(tech)

technology_counts = (
    ppi["Technology"]
    .value_counts(dropna=False)
)

print(technology_counts)



Biogas
Biomass
Coal
Coal, Diesel
Coal, Diesel, Natural Gas
Coal, Hydro, Large (>50MW)
Coal, Natural Gas
Diesel
Diesel, Geothermal
Diesel, Geothermal, Hydro, Large (>50MW), Wind
Diesel, Hydro, Large (>50MW)
Diesel, Natural Gas
Diesel, Natural Gas, Hydro, Large (>50MW)
Diesel, Natural Gas, Other
Diesel, Waste
Geothermal
Hydro, Large (>50MW)
Hydro, Large (>50MW), Diesel, N/A
Hydro, Large (>50MW), Other
Hydro, Small (<50MW)
N/A, N/A
N/A, N/A, N/A
Natural Gas
Natural Gas, Diesel
Natural Gas, Hydro, Large (>50MW), Wind
Natural Gas, N/A
Natural Gas, Other
Natural Gas, Steam
Natural Gas, Steam, Solar, CSP
Natural Gas, Steam, Solar, PV
Not Applicable
Not Applicable, Diesel, Wind
Not Applicable, N/A
Nuclear
Other
Solar, CPV
Solar, CSP
Solar, PV
Solar, PV, Biogas
Solar, PV, N/A
Solar, PV, Not Applicable
Solar, PV, Other
Solar, PV, Solar, PV
Solar, PV, Wind
Solar, PV, Wind, Coal, N/A
Solar, PV, Wind, N/A
Steam
Waste
Wind
Wind, Coal, N/A
Wind, N/A
Wind, Not Applicable
Wind, Other
Wind, Solar, PV
Te

In [7]:
technology_df = (
    ppi["Technology"]
    .value_counts(dropna=False)
    .rename_axis("Technology")
    .reset_index(name="Projects")
)

print(technology_df)

technology_df.to_csv(
    "results/gdp_visualization/Technology/technology_list.csv",
    index=False,
    encoding="utf-8-sig"
)

                                        Technology  Projects
0                                              NaN      3995
1                                        Solar, PV       925
2                                             Wind       827
3                             Hydro, Small (<50MW)       418
4                                   Not Applicable       393
5                                      Natural Gas       294
6                             Hydro, Large (>50MW)       278
7                                             Coal       241
8                                          Biomass       210
9                                           Diesel       162
10                                           Waste       133
11                                        N/A, N/A        45
12                                      Geothermal        43
13                                           Other        33
14                                          Biogas        25
15                      

Region,Country,IncomeGroup,IDA Status,Financial closure year,Financial closure Month,Project name,RelatedNames,Type of PPI,Subtype of PPI,Project status,Primary sector,Subsector,Segment,Location,ContractPeriod,GovtGrantingContract,DirectGovtSupport,DirectGovtSupportValue,InDirectGovtSupport,InDirectGovtSupportValue,Total Equity,InvestmentYear,PercentPrivate,FeesToGovernment,PhysicalAssets,TotalInvestment,CapacityType,Capacity,Technology,RelatedProjects,BidCriteria,AwardMethod,NumberOfBids,Sponsors,Sponsors Country,Main Revenue Source,Other Revenue Source,MultiLateralSupport,BiLateralSupport,TotalDebtFunding,DebtEquityGrantRatio,ProjectBanks,UnsolicitedProposal,PublicDisclosure
